In [1]:
import os
import sys
import ctypes
import resource
import time
import json
import numpy as np

# 0. CUDA HACK
oscar_cuda_path = "/oscar/rt/9.6/25/spack/x86_64_v3/cuda-12.9.0-cinrl2oeqemd3szbcakkugp2vtk2fh5t"
nvvm_lib_dir = os.path.join(oscar_cuda_path, "nvvm", "lib64")
nvrtc_lib_dir = os.path.join(oscar_cuda_path, "targets", "x86_64-linux", "lib")
standard_lib_dir = os.path.join(oscar_cuda_path, "lib64")
os.environ['CUDA_HOME'] = oscar_cuda_path
os.environ['CPATH'] = os.path.join(oscar_cuda_path, 'include')
os.environ['PATH'] = os.path.join(oscar_cuda_path, 'bin') + ":" + os.environ.get('PATH', '')
os.environ['NUMBA_CUDA_DRIVER'] = "/lib64/libcuda.so"
os.environ['LD_LIBRARY_PATH'] = f"{nvvm_lib_dir}:{nvrtc_lib_dir}:{standard_lib_dir}:/lib64:" + os.environ.get('LD_LIBRARY_PATH', '')
os.environ["IPIE_USE_GPU"] = "1"

try:
    ctypes.CDLL(os.path.join(nvvm_lib_dir, "libnvvm.so"), mode=ctypes.RTLD_GLOBAL)
except: pass

import joblib
import tensorflow as tf
from tensorflow.keras.models import load_model
from pyscf import gto, scf, lib
from ipie.utils.from_pyscf import gen_ipie_input_from_pyscf_chk
from ipie.qmc.afqmc import AFQMC
from ipie.utils.mpi import MPIHandler
import ipie.estimators.local_energy_sd
from ipie.analysis.autocorr import reblock_by_autocorr
from ipie.analysis.extraction import extract_observable

try:
    import cupy as cp
    has_cupy = True
except: has_cupy = False

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus: tf.config.experimental.set_memory_growth(gpu, True)

# CONFIG
N_ATOMS = 110
TEST_SEED = 999 
SYSTEM_NAME = f"H{N_ATOMS}"
WALKERS = 2048
NUM_BLOCKS = 50
STEPS_PER_BLOCK = 1
DEPLOY_DIR = "deployment_objects"
MODEL_PATH = os.path.join(DEPLOY_DIR, f"NN_{SYSTEM_NAME}_DeltaHF.keras")

@tf.keras.utils.register_keras_serializable()
def log_cosh_loss(y_true, y_pred):
    y_true = tf.cast(tf.reshape(y_true, tf.shape(y_pred)), dtype=y_pred.dtype)
    return tf.reduce_mean(tf.math.log(tf.math.cosh(y_pred - y_true)))

def get_dynamic_operators(mol):
    S = mol.intor('int1e_ovlp')
    h_core_ao = mol.intor('int1e_nuc') + mol.intor('int1e_kin')
    e, v = np.linalg.eigh(S)
    mask = e > 1e-15
    S_inv_sqrt = v[:, mask] @ np.diag(e[mask]**(-0.5)) @ v[:, mask].T
    S_sqrt = v[:, mask] @ np.diag(e[mask]**(0.5)) @ v[:, mask].T
    h_core_lowdin = S_inv_sqrt.T @ h_core_ao @ S_inv_sqrt
    return S_inv_sqrt, S_sqrt, h_core_lowdin, S

def estimate_nn_flops(model):
    flops = 0
    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.Dense):
            weights = layer.get_weights()
            if len(weights) > 0:
                kernel = weights[0]
                flops += 2 * kernel.shape[0] * kernel.shape[1]
                if len(weights) > 1: flops += kernel.shape[1]
    return flops

# =============================================================================
# MICRO-BENCHMARKED ML PROXY
# =============================================================================
def create_ml_local_energy_patch(ml_model, X_scaler, y_scaler, P_hf_ref, E_hf_ref, S_sqrt, h_core_dyn, C_a, use_gpu, n_atoms):
    xp = cp if use_gpu else np
    P_hf_ref_xp = xp.asarray(P_hf_ref)
    S_sqrt_xp = xp.asarray(S_sqrt)
    h_core_dyn_xp = xp.asarray(h_core_dyn)
    C_a_xp = xp.asarray(C_a)

    @tf.function(reduce_retracing=True)
    def fast_predict(inputs): return ml_model(inputs, training=False)

    # ⏱️ THE MICRO-TRACKER
    tracker = {"total_time_sec": 0.0, "calls": 0}

    def local_energy_single_det_uhf(system, hamiltonian, walkers, trial):
        # SYNC GPU BEFORE STARTING TIMER
        if use_gpu: cp.cuda.Stream.null.synchronize()
        t0 = time.perf_counter()

        nwalkers = walkers.nwalkers
        nalpha = trial.nalpha
        phi_a = walkers.phia if hasattr(walkers, 'phia') else walkers.phi[:, :, :nalpha]
        phi_b = walkers.phib if hasattr(walkers, 'phib') else walkers.phi[:, :, nalpha:]
        Psi_T_a, Psi_T_b = xp.asarray(trial.psi[:, :nalpha]), xp.asarray(trial.psi[:, nalpha:])
        phi_a, phi_b = xp.asarray(phi_a), xp.asarray(phi_b)

        O_a = xp.einsum('ui, wuj -> wij', Psi_T_a.conj(), phi_a)
        O_b = xp.einsum('ui, wuj -> wij', Psi_T_b.conj(), phi_b)
        invO_a, invO_b = xp.linalg.inv(O_a), xp.linalg.inv(O_b)
        
        G_mo_a = xp.einsum('wvi, wiu -> wvu', phi_a, xp.einsum('wij, ju -> wiu', invO_a, Psi_T_a.conj().T))
        G_mo_b = xp.einsum('wvi, wiu -> wvu', phi_b, xp.einsum('wij, ju -> wiu', invO_b, Psi_T_b.conj().T))
        
        P_ao = (xp.einsum("qi, wij, pj -> wqp", C_a_xp, G_mo_a, C_a_xp.conj()) + 
                xp.einsum("qi, wij, pj -> wqp", C_a_xp, G_mo_b, C_a_xp.conj()))

        P_lowdin = xp.einsum('ai, wib, bj -> waj', S_sqrt_xp, P_ao, S_sqrt_xp)
        delta_P = cp.asnumpy(P_lowdin - P_hf_ref_xp) if use_gpu else P_lowdin - P_hf_ref_xp
        E_1B_delta = cp.asnumpy(xp.einsum('ij, wji -> w', h_core_dyn_xp, P_lowdin - P_hf_ref_xp).real) if use_gpu else xp.einsum('ij, wji -> w', h_core_dyn_xp, P_lowdin - P_hf_ref_xp).real

        X_input = np.stack([np.real(delta_P), np.imag(delta_P)], axis=-1).astype(np.float32).reshape(nwalkers, -1)
        X_scaled = X_scaler.transform(X_input).reshape(nwalkers, n_atoms, n_atoms, 2)
        
        preds_scaled = fast_predict(X_scaled).numpy()
        E_corr_delta = y_scaler.inverse_transform(preds_scaled).flatten()

        energy_out = xp.zeros((nwalkers, 3), dtype=xp.complex128)
        energy_out[:, 0] = E_hf_ref + xp.asarray(E_1B_delta) + xp.asarray(E_corr_delta)
        energy_out[:, 1] = E_hf_ref + xp.asarray(E_1B_delta)
        energy_out[:, 2] = xp.asarray(E_corr_delta)
        
        # SYNC GPU BEFORE STOPPING TIMER TO GET TRUE EXECUTION TIME
        if use_gpu: cp.cuda.Stream.null.synchronize()
        t1 = time.perf_counter()
        
        tracker["total_time_sec"] += (t1 - t0)
        tracker["calls"] += 1

        is_gpu_walker = hasattr(walkers.weight, 'device') or 'cupy' in str(type(walkers.weight))
        return cp.asarray(energy_out) if (is_gpu_walker and use_gpu) else cp.asnumpy(energy_out) if use_gpu else energy_out

    return local_energy_single_det_uhf, tracker

# MAIN EXECUTION
comm = MPIHandler()
rank = comm.rank
if has_cupy and cp.cuda.runtime.getDeviceCount() > 0: cp.cuda.Device(rank % cp.cuda.runtime.getDeviceCount()).use()

if rank == 0:
    ml_model = load_model(MODEL_PATH, custom_objects={"log_cosh_loss": log_cosh_loss})
    X_scaler = joblib.load(os.path.join(DEPLOY_DIR, "X_scaler.save"))
    y_scaler = joblib.load(os.path.join(DEPLOY_DIR, "y_scaler.save"))
    P_hf_ref = np.load(os.path.join(DEPLOY_DIR, "P_hf_lowdin.npy"))
    
    mol = gto.M(atom=[("H", 0.74 * j, 0, 0) for j in range(N_ATOMS)], basis="sto-6g", verbose=0)
    mf = scf.UHF(mol)
    mf.kernel()
    gen_ipie_input_from_pyscf_chk(mf.chkfile, hamil_file=f"ham_h{N_ATOMS}.h5", wfn_file=f"wfn_h{N_ATOMS}.h5", verbose=0, chol_cut=1e-5)
    E_HF = mf.e_tot
    C_a = mf.mo_coeff[0] if np.ndim(mf.mo_coeff) == 3 else mf.mo_coeff
    _, S_sqrt, h_core, _ = get_dynamic_operators(mol)
else: E_HF = C_a = S_sqrt = h_core = ml_model = X_scaler = y_scaler = P_hf_ref = None

E_HF, C_a, S_sqrt, h_core = comm.comm.bcast(E_HF, root=0), comm.comm.bcast(C_a, root=0), comm.comm.bcast(S_sqrt, root=0), comm.comm.bcast(h_core, root=0)

afqmc = AFQMC.build_from_hdf5(num_elec=(N_ATOMS//2, N_ATOMS//2), ham_file=f"ham_h{N_ATOMS}.h5", wfn_file=f"wfn_h{N_ATOMS}.h5", num_walkers=WALKERS, num_blocks=NUM_BLOCKS, num_steps_per_block=STEPS_PER_BLOCK, verbose=0, seed=TEST_SEED)
if has_cupy: afqmc.cuda = True
afqmc.mpi_handler = comm

# Fixed: Always calculate and set nwalkers regardless of comm.size
local_walkers = WALKERS // comm.size
if rank < (WALKERS % comm.size): local_walkers += 1
afqmc.nwalkers = local_walkers

ml_proxy, loop_tracker = create_ml_local_energy_patch(ml_model, X_scaler, y_scaler, P_hf_ref, E_HF, S_sqrt, h_core, C_a, has_cupy, N_ATOMS)

# Apply patches
targets = [getattr(ipie.estimators.local_energy_sd, f) for f in ["local_energy_single_det_uhf", "local_energy_single_det_batch_gpu", "local_energy_single_det_uhf_batch_gpu", "local_energy_single_det_uhf_batch"] if hasattr(ipie.estimators.local_energy_sd, f)]
for mod_name, module in list(sys.modules.items()):
    if module and mod_name.startswith("ipie"):
        try:
            for attr_name, attr_value in vars(module).items():
                if attr_value in targets: setattr(module, attr_name, ml_proxy)
        except: pass
if hasattr(afqmc, 'propagator'): afqmc.propagator.local_energy = ml_proxy
if hasattr(afqmc, 'estimators'):
    try: afqmc.estimators['energy'].local_energy = ml_proxy
    except: pass

afqmc.run()

if rank == 0:
    # Get Final Reblocked Energy
    estimator_filename = afqmc.estimators.filename
    qmc_data = extract_observable(estimator_filename, "energy")
    df_ac = reblock_by_autocorr(qmc_data["ETotal"][1:], verbose=0)
    final_energy, final_error = float(df_ac["ETotal_ac"].iloc[0]), float(df_ac["ETotal_error_ac"].iloc[0])

    # ⏱️ Micro-Benchmark Calculations
    avg_loop_time = loop_tracker["total_time_sec"] / loop_tracker["calls"] if loop_tracker["calls"] > 0 else 0
    
    flops_physics_per_walker = 8 * (N_ATOMS ** 3) 
    flops_nn_per_walker = estimate_nn_flops(ml_model)
    flops_total_per_call = (flops_physics_per_walker + flops_nn_per_walker) * afqmc.nwalkers
    loop_tflops = (flops_total_per_call / avg_loop_time) / 1e12 if avg_loop_time > 0 else 0

    metrics = {
        "system": SYSTEM_NAME, "backend": "ML Proxy",
        "results": {"final_energy_ha": final_energy, "final_error_ha": final_error},
        "micro_benchmark": {
            "total_calls": loop_tracker["calls"],
            "avg_time_per_call_sec": round(avg_loop_time, 6),
            "flops_per_call": flops_total_per_call,
            "throughput_tflops": round(loop_tflops, 6)
        },
        "memory": {
            "peak_gpu_vram_mb": round(cp.get_default_memory_pool().total_bytes() / (1024**2) + (tf.config.experimental.get_memory_info('GPU:0')['peak'] / 1024**2 if gpus else 0), 2)
        }
    }
    with open(f"micro_metrics_ML_{SYSTEM_NAME}.json", "w") as f: json.dump(metrics, f, indent=4)
    print(json.dumps(metrics, indent=4))

2026-03-02 11:51:58.605034: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1772470327.717078 2065814 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 43606 MB memory:  -> device: 0, name: NVIDIA A40, pci bus id: 0000:23:00.0, compute capability: 8.6


# random seed is 999
            Block                   Weight            WeightFactor            HybridEnergy                  ENumer                  EDenom                  ETotal                  E1Body                  E2Body
                0   2.0480000000000000e+03  2.0480000000000000e+03  0.0000000000000000e+00 -1.1276373524564177e+05  2.0480000000000000e+03 -5.5060417600411022e+01 -5.4901538767355525e+01 -1.5887883305549622e-01
                1   2.1993988629961746e+03  2.0480000000000000e+03 -2.8533313437990053e+01 -1.2111581165492171e+05  2.1993988629961746e+03 -5.5067688581928834e+01 -5.4901526385748298e+01 -1.6616219618053596e-01
                2   2.1997230435705333e+03  2.0480000000000000e+03 -2.8592036597407084e+01 -1.2114974371414738e+05  2.1997230435705333e+03 -5.5074998676879005e+01 -5.4901542393240582e+01 -1.7345628363842053e-01
                3   2.1992296038420445e+03  2.0480000000000000e+03 -2.8502643589277195e+01 -1.2113912863832401e+05  2.1992296038420445e

In [2]:
(-55.16727159818634 - -55.231048838619365)*627.5095

40.020824255508145